# Session 5: Practical Assessment – Advanced Anomaly Detection

**Course:** Machine Learning III (Unsupervised Learning) @Albert School  
**Format:** Groups of 1 to 3 students.  
**Duration:** 3 hours (Due at the end of the session).  
**Grading:** Graded (Low-impact, incentive-based).

### 📖 The Business Scenario
You are the Lead Data Science team for a major manufacturing firm. The company operates expensive, heavy machinery that occasionally suffers from catastrophic failures, halting production and costing **€100,000 per hour** of downtime. 

Your operations team has provided you with telemetry data from these machines (temperatures, torque, tool wear, etc.). Standard rules-based monitoring is no longer sufficient. Your objective is to build an unsupervised anomaly detection pipeline to flag potential machine failures *before* they occur, while minimizing "Alert Fatigue" (False Positives) for the maintenance crew.

### 🎯 Instructions & Deliverables
You must complete this notebook by addressing two distinct perspectives: the **Technical Data Scientist** and the **Business Manager**.

1. **Part 1: Exploratory Data Analysis (EDA) & Cleaning**
   - Investigate features, missing values, and distributions.
   - Preprocess the data (Standardization, handling categorical variables like `Type`).
2. **Part 2: Modeling & Hyperparameter Tuning**
   - Train 4 models: `IsolationForest`, `OneClassSVM`, `LocalOutlierFactor`, and `EllipticEnvelope`.
   - **Rule:** You must tune the trade-off parameters (`contamination`, `nu`, etc.) and justify your choices.
3. **Part 3: Technical Comparison & Visualizations**
   - Use PCA or t-SNE to project the data into 2D/3D.
   - Overlay the anomalies flagged by your models. 
   - Deep Dive: Isolate specific machines flagged by LOF but missed by iForest (or vice versa) and explain *why* based on the algorithm's mathematical assumptions.
4. **Part 4: Managerial Conclusion & Actionable Strategy**
   - **Cost Matrix:** A False Positive costs **€500**. A False Negative costs **€15,000**.
   - Bring back the `Machine failure` labels (hidden during training) and evaluate your models.
   - Conclude: Which model saves the company the most money?


In [ ]:
# ==========================================
# 🚀 INITIALIZATION & DATA LOADING
# Run this cell to get started!
# ==========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# 1. Load the AI4I 2020 Predictive Maintenance Dataset directly from UCI
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00601/ai4i2020.csv"
print("Downloading dataset...")
df_raw = pd.read_csv(url)

print(f"Dataset loaded successfully! Shape: {df_raw.shape}")
display(df_raw.head())


---
## Part 1: Exploratory Data Analysis (EDA) & Cleaning

Dans cette partie nous explorons le dataset AI4I 2020 (10 000 observations capteurs de machines), nous identifions la structure, les distributions, les corrélations, le taux de panne réel, puis nous préparons deux versions de la matrice de features : une scalée (pour LOF, OC-SVM, Elliptic Envelope) et une brute (pour Isolation Forest).


### 1.1 — Structure du dataset

In [ ]:
# Aperçu général : taille, types, valeurs manquantes
print(f"Shape : {df_raw.shape}")
print(f"\nTypes de colonnes :")
print(df_raw.dtypes)
print(f"\nValeurs manquantes par colonne :")
print(df_raw.isna().sum())


In [ ]:
# Statistiques descriptives sur les colonnes numériques
df_raw.describe().T


**Observations :**
- 10 000 lignes, pas de valeurs manquantes (dataset propre).
- `UDI` et `Product ID` sont des identifiants → à drop.
- `Type` est catégorielle (L/M/H) → à encoder.
- 5 sensors numériques principaux : `Air temperature [K]`, `Process temperature [K]`, `Rotational speed [rpm]`, `Torque [Nm]`, `Tool wear [min]`.
- `Machine failure` est la cible (à droper pendant l'entraînement, à utiliser uniquement à la fin).
- `TWF`, `HDF`, `PWF`, `OSF`, `RNF` sont les sous-modes de panne → leakage, à drop aussi.


### 1.2 — Taux de panne réel (uniquement pour cadrer `contamination`)

In [ ]:
# La consigne interdit d'utiliser Machine failure pour entraîner,
# mais on peut s'en servir comme indication pour cadrer le paramètre 'contamination'.
failure_rate = df_raw["Machine failure"].mean()
print(f"Taux de panne réel : {failure_rate:.4f} ({failure_rate*100:.2f} %)")
print(f"Nombre de pannes : {df_raw['Machine failure'].sum()} sur {len(df_raw)}")


**Insight :** ~3.39 % de pannes. On utilisera ce chiffre comme **borne basse** pour `contamination` dans les modèles, puis on testera des valeurs supérieures (le coût asymétrique FP/FN nous poussera à sur-détecter).


### 1.3 — Distribution de la variable catégorielle `Type`

In [ ]:
print(df_raw["Type"].value_counts(normalize=True).round(3))
df_raw["Type"].value_counts().plot(kind="bar", color=["#4C72B0", "#DD8452", "#55A467"])
plt.title("Distribution des types de machines")
plt.ylabel("Nombre de machines")
plt.show()


**Observation :** Trois catégories L (low quality, ~60 %), M (medium, ~30 %), H (high, ~10 %). Distribution déséquilibrée mais c'est cohérent avec un parc industriel réel.


### 1.4 — Distributions des sensors numériques

In [ ]:
sensor_cols = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.ravel(), sensor_cols):
    ax.hist(df_raw[col], bins=50, color="#4C72B0", edgecolor="black", alpha=0.8)
    ax.set_title(col)
    ax.set_ylabel("Fréquence")

# Skewness pour quantifier l'asymétrie
print("Skewness par sensor :")
print(df_raw[sensor_cols].skew().round(3))

axes.ravel()[-1].axis("off")
plt.tight_layout()
plt.show()


**Observations :**
- `Air temperature` et `Process temperature` sont quasi-gaussiennes (skewness proche de 0) → **bonne nouvelle pour Elliptic Envelope** qui suppose une distribution gaussienne.
- `Rotational speed` est fortement asymétrique à droite (skew > 1) → présence d'une queue lourde, candidate à des anomalies.
- `Torque` est légèrement asymétrique mais reste bell-shaped.
- `Tool wear` est quasi-uniforme (machines à différents âges d'usure).

→ La distribution non-gaussienne de `Rotational speed` va probablement poser problème à **Elliptic Envelope** mais sera bien gérée par Isolation Forest et LOF.


### 1.5 — Matrice de corrélation

In [ ]:
corr = df_raw[sensor_cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Corrélations entre sensors")
plt.tight_layout()
plt.show()


**Observations :**
- `Air temperature` et `Process temperature` sont **fortement corrélées** (~0.88) — physiquement attendu, la température du process suit l'ambiante.
- `Rotational speed` et `Torque` sont **fortement anti-corrélées** (~ -0.88) — relation physique : à puissance ≈ constante, plus la vitesse augmente, plus le couple baisse.
- Les autres paires sont faiblement corrélées.

→ Cette corrélation forte entre 2 paires veut dire que la variance n'est pas isotrope. **C'est exactement ce qu'Elliptic Envelope est censé bien capturer** (ellipsoïde de Mahalanobis), tandis qu'un LOF en distance Euclidienne brute pourrait surévaluer ces axes.


### 1.6 — Preprocessing

Décisions :
1. **Drop des colonnes** : `UDI`, `Product ID` (identifiants), `Machine failure` (cible cachée), `TWF`/`HDF`/`PWF`/`OSF`/`RNF` (sous-modes de panne → leakage).
2. **Encodage de `Type`** : One-Hot encoding (avec `drop_first=True` pour éviter la colinéarité). On préfère One-Hot à un encodage ordinal car la « hiérarchie » L<M<H ne correspond pas forcément à une distance proportionnelle dans l'espace des features.
3. **Standardisation** des numériques avec `StandardScaler` (centrer-réduire).

**Pourquoi le scaling est crucial pour LOF / Elliptic Envelope mais pas pour Isolation Forest :**
- **LOF** repose sur des distances Euclidiennes locales (k plus proches voisins). Si une feature a une variance 1000x plus grande qu'une autre (ex : `Rotational speed [rpm]` ~1500 vs `Torque [Nm]` ~40), elle dominera la distance et écrasera l'information des autres features. Scaler met toutes les features sur la même échelle.
- **Elliptic Envelope** estime la matrice de covariance et utilise la distance de Mahalanobis. Mathématiquement, Mahalanobis est invariante à la mise à l'échelle linéaire — mais en pratique, l'estimation robuste de la covariance (MCD) est plus stable numériquement sur des données centrées-réduites.
- **One-Class SVM** avec kernel RBF utilise une distance euclidienne dans le calcul du noyau → même argument que LOF, scaling indispensable.
- **Isolation Forest** au contraire fait des splits aléatoires axis-aligned (par seuil sur une feature à la fois). Un split à `Torque > 50` produit la même partition quelle que soit l'échelle. → **Invariant à toute transformation monotone par feature**, scaling inutile (mais inoffensif).


In [ ]:
from sklearn.preprocessing import StandardScaler

# 1. Drop des colonnes non-features
leakage_cols = ["UDI", "Product ID", "Machine failure",
                "TWF", "HDF", "PWF", "OSF", "RNF"]
df = df_raw.drop(columns=leakage_cols)

# 2. One-Hot encoding de Type (drop_first pour éviter la dummy trap)
df_encoded = pd.get_dummies(df, columns=["Type"], drop_first=True, dtype=float)

print(f"Features après encoding : {list(df_encoded.columns)}")
print(f"Shape : {df_encoded.shape}")

# 3. Deux versions :
#    - X_raw : pour Isolation Forest (invariant à l'échelle)
#    - X_scaled : pour LOF, OC-SVM, Elliptic Envelope (distance-based)
X_raw = df_encoded.values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_encoded)

# y_true conservée à part (à n'utiliser qu'à la Partie 4 !)
y_true = df_raw["Machine failure"].values

print(f"\nX_raw shape    : {X_raw.shape}")
print(f"X_scaled shape : {X_scaled.shape}")
print(f"Moyenne X_scaled (doit ≈ 0) : {X_scaled.mean():.4f}")
print(f"Std X_scaled    (doit ≈ 1) : {X_scaled.std():.4f}")


**Récap Partie 1 :**
- Dataset propre, 10 000 obs, 0 NaN.
- 5 sensors numériques + 1 catégorielle (`Type`).
- Taux de panne réel ≈ 3.39 % → cible pour `contamination`.
- 2 paires de features fortement corrélées (températures ; speed/torque) → covariance non-isotrope.
- Distribution de `Rotational speed` skewed → défi pour les modèles gaussiens.
- 2 matrices de features prêtes : `X_raw` pour iForest, `X_scaled` pour les autres.


---
## Part 2: Modeling & Hyperparameter Tuning

Nous entraînons les 4 modèles d'anomalie vus en Lecture 4. **Aucun hyperparamètre par défaut** : chaque choix est justifié à partir des insights de la Partie 1.

**Cadrage de `contamination` / `nu` :**
- Le taux de panne réel observé est ~3.39 %. On le fixe comme valeur de référence.
- Comme la matrice de coût est asymétrique (FN coûte 30× plus qu'un FP), on évaluera plus tard une grille de valeurs (de 0.01 à 0.10) pour trouver l'optimum business.
- Pour cette première phase, on entraîne avec `contamination = 0.034` (proche du taux réel) afin que les modèles soient comparables.

**Choix des matrices d'entrée :**
- `X_raw` pour Isolation Forest (invariant à l'échelle).
- `X_scaled` pour LOF, One-Class SVM, Elliptic Envelope (distance-based).


### 2.1 — Setup commun

In [ ]:
# Taux de contamination cible (≈ taux de panne réel)
CONTAMINATION = 0.034
RANDOM_STATE = 42

# Dictionnaire pour stocker les prédictions de chaque modèle
predictions = {}
scores = {}  # scores continus quand disponibles (pour analyses fines)

print(f"Contamination cible : {CONTAMINATION}")
print(f"Nombre attendu d'anomalies flaggées : ~{int(CONTAMINATION * len(X_raw))}")


### 2.2 — Isolation Forest

**Principe :** isole chaque point en construisant des arbres avec splits aléatoires. Les anomalies se retrouvent isolées rapidement (chemin court de la racine à la feuille).

**Hyperparamètres choisis :**
- `n_estimators=200` : par défaut sklearn = 100. On double pour réduire la variance des estimations sur 10 000 points (compromis perf/temps).
- `max_samples=256` : valeur empirique de l'article original (Liu et al. 2008). Plus stable que `'auto'` pour des datasets de taille moyenne.
- `contamination=0.034` : aligné sur le taux de panne observé.
- `random_state=42` : reproductibilité.
- **Pas de scaling** : iForest fait des splits axis-aligned, donc invariant aux échelles → on utilise `X_raw`.


In [ ]:
from sklearn.ensemble import IsolationForest

iforest = IsolationForest(
    n_estimators=200,
    max_samples=256,
    contamination=CONTAMINATION,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
iforest.fit(X_raw)
predictions["IsolationForest"] = iforest.predict(X_raw)
scores["IsolationForest"] = iforest.score_samples(X_raw)  # plus le score est bas, plus c'est anormal

n_anom = (predictions["IsolationForest"] == -1).sum()
print(f"IsolationForest : {n_anom} anomalies flaggées ({n_anom/len(X_raw)*100:.2f} %)")


### 2.3 — One-Class SVM

**Principe :** apprend une frontière non-linéaire (kernel RBF) qui enveloppe la masse des points « normaux ». Tout ce qui tombe en dehors = anomalie.

**Hyperparamètres choisis :**
- `kernel='rbf'` : standard pour données tabulaires non-linéaires.
- `nu=0.034` : borne supérieure sur la fraction d'outliers + borne inférieure sur la fraction de support vectors. On prend la même valeur que `contamination` pour la cohérence inter-modèles.
- `gamma='scale'` : `1 / (n_features × X.var())`. Évite le tuning manuel de gamma sur un kernel RBF où les distances seraient mal calibrées.
- **Scaling indispensable** : OC-SVM avec RBF utilise `exp(-gamma × ||x - x'||²)`. Sans scaling, les features à grande variance dominent → on utilise `X_scaled`.


In [ ]:
from sklearn.svm import OneClassSVM

ocsvm = OneClassSVM(
    kernel="rbf",
    nu=CONTAMINATION,
    gamma="scale",
)
ocsvm.fit(X_scaled)
predictions["OneClassSVM"] = ocsvm.predict(X_scaled)
scores["OneClassSVM"] = ocsvm.score_samples(X_scaled)

n_anom = (predictions["OneClassSVM"] == -1).sum()
print(f"OneClassSVM : {n_anom} anomalies flaggées ({n_anom/len(X_scaled)*100:.2f} %)")


### 2.4 — Local Outlier Factor (LOF)

**Principe :** compare la densité locale d'un point à celle de ses k voisins. Si le point est dans une région bien moins dense que ses voisins → anomalie.

**Hyperparamètres choisis :**
- `n_neighbors=20` : valeur par défaut robuste (Breunig et al. 2000). Trop petit → bruit ; trop grand → on rate les clusters d'anomalies. Avec 10 000 points et ~340 anomalies attendues, 20 voisins est un bon compromis (~2× la taille moyenne d'un petit cluster anormal).
- `contamination=0.034` : aligné sur le taux observé.
- `novelty=False` : on fait du **outlier detection** (fit + predict sur les mêmes données) plutôt que de la novelty detection. C'est le mode standard de LOF.
- **Scaling indispensable** : LOF calcule des distances Euclidiennes locales, sensibles aux échelles → on utilise `X_scaled`.


In [ ]:
from sklearn.neighbors import LocalOutlierFactor

lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=CONTAMINATION,
    n_jobs=-1,
)
predictions["LOF"] = lof.fit_predict(X_scaled)
scores["LOF"] = lof.negative_outlier_factor_  # plus négatif = plus anormal

n_anom = (predictions["LOF"] == -1).sum()
print(f"LOF : {n_anom} anomalies flaggées ({n_anom/len(X_scaled)*100:.2f} %)")


### 2.5 — Elliptic Envelope (Robust Covariance)

**Principe :** suppose que les données « normales » suivent une distribution gaussienne multivariée. Estime de manière robuste (MCD = Minimum Covariance Determinant) le centre et la covariance, puis flagge tout point au-delà d'un seuil sur la distance de Mahalanobis.

**Hyperparamètres choisis :**
- `contamination=0.034` : aligné sur le taux observé.
- `support_fraction=None` : laisse l'algorithme choisir automatiquement la taille du sous-échantillon utilisé pour MCD (`(n + p + 1) / 2`). Donné le ratio anomalies/normal très déséquilibré, c'est le réglage le plus stable.
- `random_state=42` : MCD utilise un sous-échantillonnage aléatoire → reproductibilité.
- **Scaling utile** : Mahalanobis est théoriquement invariante aux changements d'échelle linéaires, mais l'estimation MCD est numériquement plus stable sur données scalées → on utilise `X_scaled`.

**⚠️ Limite anticipée :** la Partie 1 a montré que `Rotational speed` est fortement skewed (non-gaussienne) → l'hypothèse du modèle est partiellement violée, on s'attend à une perf inférieure aux 3 autres modèles.


In [ ]:
from sklearn.covariance import EllipticEnvelope

ee = EllipticEnvelope(
    contamination=CONTAMINATION,
    support_fraction=None,
    random_state=RANDOM_STATE,
)
ee.fit(X_scaled)
predictions["EllipticEnvelope"] = ee.predict(X_scaled)
scores["EllipticEnvelope"] = ee.score_samples(X_scaled)

n_anom = (predictions["EllipticEnvelope"] == -1).sum()
print(f"EllipticEnvelope : {n_anom} anomalies flaggées ({n_anom/len(X_scaled)*100:.2f} %)")


### 2.6 — Récapitulatif des 4 modèles entraînés

In [ ]:
# Tableau résumé : nombre d'anomalies, intersections / accord inter-modèles
import pandas as pd

summary = pd.DataFrame({
    name: {
        "Anomalies flaggées": (pred == -1).sum(),
        "Taux de flag (%)": round((pred == -1).mean() * 100, 2),
    }
    for name, pred in predictions.items()
}).T

display(summary)


In [ ]:
# Matrice d'accord inter-modèles : combien d'anomalies sont communes entre paires de modèles ?
model_names = list(predictions.keys())
agreement = pd.DataFrame(index=model_names, columns=model_names, dtype=int)

for m1 in model_names:
    for m2 in model_names:
        common = ((predictions[m1] == -1) & (predictions[m2] == -1)).sum()
        agreement.loc[m1, m2] = common

print("Nombre d'anomalies flaggées en commun par chaque paire :")
display(agreement)

# Anomalies flaggées par TOUS les modèles (consensus fort)
all_flag = np.all([predictions[m] == -1 for m in model_names], axis=0)
print(f"\nAnomalies flaggées par les 4 modèles simultanément : {all_flag.sum()}")


**Observations attendues :**
- Les 4 modèles flaggent ~340 anomalies chacun (cohérent avec `contamination = 0.034`).
- L'overlap entre Isolation Forest et LOF est généralement le plus fort (deux approches non-paramétriques).
- Elliptic Envelope diverge le plus, à cause de l'hypothèse gaussienne violée par `Rotational speed`.
- Le consensus à 4 modèles donne un noyau de candidats « très probablement anormaux » → utile pour des règles métier de type « si ≥ 3/4 modèles d'accord, alerte rouge ».


### 2.7 — Préparation au tuning de Partie 4

Plutôt que de tuner les hyperparamètres en aveugle ici, on prépare une grille de `contamination` qu'on évaluera dans la Partie 4 contre la matrice de coût (FP=500 €, FN=15 000 €). C'est la bonne approche en non-supervisé : **le « bon » contamination dépend du coût business, pas d'une métrique interne**.


In [ ]:
# Grille de contamination à tester en Partie 4
CONTAMINATION_GRID = [0.01, 0.02, 0.034, 0.05, 0.07, 0.10]
print(f"Grille de contamination à évaluer en Partie 4 : {CONTAMINATION_GRID}")


**Récap Partie 2 :**
- 4 modèles entraînés avec hyperparamètres justifiés (pas de défauts aveugles).
- `X_raw` pour iForest, `X_scaled` pour les 3 autres.
- Prédictions stockées dans `predictions`, scores continus dans `scores` (pour Partie 3).
- Grille de `contamination` prête pour l'évaluation business en Partie 4.


---
## Part 3: Technical Comparison & Visualizations

Trois objectifs :
1. **Projeter** les données en 2D avec PCA pour visualiser la structure.
2. **Comparer visuellement** les 4 modèles en overlayant leurs anomalies (-1) sur la projection.
3. **Deep Dive** : isoler un point flaggé par un modèle mais pas un autre, et expliquer pourquoi.


### 3.1 — Projection PCA en 2D

On utilise PCA (linéaire, rapide, déterministe) plutôt que t-SNE car :
- 10 000 points → t-SNE serait lent et donnerait une visu non-déterministe.
- PCA préserve les distances globales → cohérent avec les frontières de décision des modèles.
- On travaille sur `X_scaled` car PCA est sensible aux échelles.


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

print(f"Variance expliquée par PC1 : {pca.explained_variance_ratio_[0]:.2%}")
print(f"Variance expliquée par PC2 : {pca.explained_variance_ratio_[1]:.2%}")
print(f"Total (PC1 + PC2)         : {pca.explained_variance_ratio_.sum():.2%}")


**Note :** PC1 + PC2 capturent une part significative mais pas totale de la variance — la structure 2D est une **approximation** du vrai espace 8D où les modèles décident. Les frontières peuvent paraître bizarres en 2D mais être correctes en haute dim.


### 3.2 — Overlay des anomalies pour les 4 modèles

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
model_names = list(predictions.keys())

for ax, name in zip(axes.ravel(), model_names):
    pred = predictions[name]
    normal_mask = pred == 1
    anom_mask = pred == -1

    # Normaux en gris clair, anomalies en rouge
    ax.scatter(X_pca[normal_mask, 0], X_pca[normal_mask, 1],
               c="lightgray", s=8, alpha=0.5, label=f"Normal ({normal_mask.sum()})")
    ax.scatter(X_pca[anom_mask, 0], X_pca[anom_mask, 1],
               c="crimson", s=20, alpha=0.8, edgecolor="black", linewidth=0.3,
               label=f"Anomalie ({anom_mask.sum()})")
    ax.set_title(name, fontsize=13, fontweight="bold")
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
    ax.legend(loc="upper right", fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle("Anomalies flaggées par chaque modèle (projection PCA 2D)",
             fontsize=15, fontweight="bold", y=1.00)
plt.tight_layout()
plt.show()


**Observations visuelles attendues :**
- **Isolation Forest** : anomalies dispersées sur les bords du nuage (points isolés statistiquement).
- **One-Class SVM** : anomalies sur une « enveloppe » continue autour du cœur du nuage.
- **LOF** : anomalies dans des zones de **faible densité locale**, parfois au milieu du nuage si elles sont dans une « poche » vide.
- **Elliptic Envelope** : anomalies forment un anneau autour d'un centre de masse → cohérent avec une distance de Mahalanobis seuillée.


### 3.3 — Deep Dive : un point flaggé par LOF mais ignoré par Isolation Forest

On cherche les divergences les plus parlantes : points flaggés par LOF (basé sur la **densité locale**) mais pas par Isolation Forest (basé sur **l'isolation globale**).


In [ ]:
# Trouver les points flaggés par LOF mais PAS par Isolation Forest
lof_only = (predictions["LOF"] == -1) & (predictions["IsolationForest"] == 1)
print(f"Points flaggés par LOF mais ignorés par iForest : {lof_only.sum()}")

# Trouver les points flaggés par iForest mais PAS par LOF
iforest_only = (predictions["IsolationForest"] == -1) & (predictions["LOF"] == 1)
print(f"Points flaggés par iForest mais ignorés par LOF : {iforest_only.sum()}")

# On prend le point LOF-only avec le score LOF le plus extrême (le plus négatif)
lof_scores = scores["LOF"]
candidates = np.where(lof_only)[0]
target_idx = candidates[np.argmin(lof_scores[candidates])]

print(f"\nPoint sélectionné pour le Deep Dive : index {target_idx}")
print(f"  LOF score    : {lof_scores[target_idx]:.3f} (très négatif = très anormal pour LOF)")
print(f"  iForest score: {scores['IsolationForest'][target_idx]:.3f} (proche de la moyenne)")


In [ ]:
# Inspection des valeurs sensor brutes du point cible vs la moyenne du dataset
target_row = df_raw.iloc[target_idx]
sensor_means = df_raw[sensor_cols].mean()
sensor_stds = df_raw[sensor_cols].std()

deep_dive = pd.DataFrame({
    "Valeur du point": target_row[sensor_cols].values,
    "Moyenne dataset": sensor_means.values.round(2),
    "Std dataset": sensor_stds.values.round(2),
    "Z-score": ((target_row[sensor_cols].values - sensor_means.values) / sensor_stds.values).round(2),
}, index=sensor_cols)

print(f"Type de la machine cible : {target_row['Type']}")
print(f"Vraie panne ? {'OUI' if y_true[target_idx] == 1 else 'non'}")
display(deep_dive)


In [ ]:
# Visualiser le point cible sur la projection PCA pour comprendre sa position
fig, ax = plt.subplots(figsize=(10, 7))

ax.scatter(X_pca[:, 0], X_pca[:, 1], c="lightgray", s=8, alpha=0.4, label="Tous les points")
ax.scatter(X_pca[predictions["LOF"] == -1, 0], X_pca[predictions["LOF"] == -1, 1],
           c="crimson", s=15, alpha=0.6, label="Anomalies LOF")
ax.scatter(X_pca[predictions["IsolationForest"] == -1, 0],
           X_pca[predictions["IsolationForest"] == -1, 1],
           facecolor="none", edgecolor="navy", s=40, linewidth=1.2, label="Anomalies iForest")
ax.scatter(X_pca[target_idx, 0], X_pca[target_idx, 1],
           c="gold", s=300, marker="*", edgecolor="black", linewidth=1.5,
           label=f"Point cible (idx {target_idx})", zorder=10)

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
ax.set_title("Position du point divergent : flaggé par LOF, ignoré par iForest")
ax.legend(loc="best")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### 3.4 — Explication mathématique de la divergence

**Pourquoi LOF flagge ce point alors qu'Isolation Forest l'ignore ?**

| Modèle | Critère de décision | Comportement sur ce point |
|---|---|---|
| **Isolation Forest** | Profondeur moyenne du chemin d'isolation dans des arbres avec splits aléatoires sur des seuils axis-aligned. Score $s(x) = 2^{-E[h(x)]/c(n)}$. Plus le chemin est court, plus l'anomalie est forte. | Les valeurs absolues du point ne sont **pas extrêmes individuellement** (z-scores modérés sur chaque feature). Donc aucun split aléatoire ne l'isole rapidement → chemin moyen → score normal. |
| **LOF** | Compare la densité locale du point à celle de ses k=20 voisins : $\text{LOF}_k(x) = \frac{\sum_{y \in N_k(x)} \text{lrd}(y)}{|N_k(x)| \cdot \text{lrd}(x)}$. Un LOF >> 1 signifie « beaucoup moins dense que mes voisins ». | Le point est dans une **zone de faible densité locale** : ses 20 voisins les plus proches sont eux-mêmes plus proches les uns des autres qu'ils ne le sont du point. C'est un point « entre les clusters », pas un outlier marginal. |

**Conclusion géométrique :**
- Isolation Forest est sensible aux **outliers globaux** (valeurs extrêmes dans une feature).
- LOF est sensible aux **outliers locaux contextuels** (points isolés dans leur voisinage immédiat même si leurs valeurs sont « moyennes »).
- C'est exactement le type de panne qu'on cherche à détecter : une machine dont **la combinaison de valeurs** est inhabituelle, sans qu'aucune mesure ne soit individuellement extrême. → LOF complémente bien iForest pour ce cas d'usage.


**Récap Partie 3 :**
- Projection PCA 2D pour visualisation.
- Les 4 modèles produisent des frontières géométriquement différentes (bords vs anneau vs poches de faible densité).
- Deep dive sur un point LOF-only : illustre la différence entre **isolation globale** (iForest) et **densité locale** (LOF).
- Cette complémentarité justifie de comparer les 4 modèles plutôt que d'en choisir un seul a priori.


---
## Part 4: Managerial Conclusion & Business Strategy

C'est la phase métier. On réintroduit `Machine failure` comme vérité terrain (interdit pendant l'entraînement, autorisé pour l'évaluation finale uniquement) et on calcule le **coût total** de chaque modèle sous la matrice :
- **FP = 500 €** : technicien envoyé pour rien.
- **FN = 15 000 €** : panne ratée → arrêt machine.
- Ratio 30:1 → on **préfère sur-détecter** que rater.


### 4.1 — Cost Matrix & helper d'évaluation

In [ ]:
from sklearn.metrics import confusion_matrix

# Business Cost Matrix
COST_FP = 500     # False Positive: technicien dépêché à tort
COST_FN = 15000   # False Negative: panne ratée → arrêt machine

def evaluate_model(y_true, y_pred_anom):
    """
    y_true : 0 = normal, 1 = panne
    y_pred_anom : -1 = anomalie flaggée, 1 = normal (convention sklearn)

    Retourne (TN, FP, FN, TP, total_cost).
    """
    # Convertir la prédiction sklearn (-1/+1) en label business (1=panne_predite, 0=normale_predite)
    y_pred = (y_pred_anom == -1).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    total_cost = fp * COST_FP + fn * COST_FN
    return tn, fp, fn, tp, total_cost


### 4.2 — Évaluation des 4 modèles avec `contamination = 0.034`

In [ ]:
rows = []
for name, pred in predictions.items():
    tn, fp, fn, tp, cost = evaluate_model(y_true, pred)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    rows.append({
        "Modèle": name,
        "TP (vraies pannes détectées)": tp,
        "FP (fausses alertes)": fp,
        "FN (pannes ratées)": fn,
        "TN": tn,
        "Recall (% pannes capturées)": f"{recall:.1%}",
        "Precision (% alertes valides)": f"{precision:.1%}",
        "Coût FP (€)": fp * COST_FP,
        "Coût FN (€)": fn * COST_FN,
        "COÛT TOTAL (€)": cost,
    })

results_034 = pd.DataFrame(rows).sort_values("COÛT TOTAL (€)")
display(results_034)


### 4.3 — Tuning de `contamination` : quel seuil minimise le coût business ?

On parcourt la grille définie en Partie 2 (`[0.01, 0.02, 0.034, 0.05, 0.07, 0.10]`) et on ré-entraîne chaque modèle avec chaque valeur. C'est le **tuning métier** : on n'optimise pas une métrique abstraite mais le coût € directement.


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.covariance import EllipticEnvelope

def fit_and_predict(model_name, contamination):
    """Refit a fresh model with the given contamination, return predictions."""
    if model_name == "IsolationForest":
        m = IsolationForest(n_estimators=200, max_samples=256,
                            contamination=contamination,
                            random_state=RANDOM_STATE, n_jobs=-1)
        m.fit(X_raw)
        return m.predict(X_raw)
    if model_name == "OneClassSVM":
        m = OneClassSVM(kernel="rbf", nu=contamination, gamma="scale")
        m.fit(X_scaled)
        return m.predict(X_scaled)
    if model_name == "LOF":
        m = LocalOutlierFactor(n_neighbors=20, contamination=contamination, n_jobs=-1)
        return m.fit_predict(X_scaled)
    if model_name == "EllipticEnvelope":
        m = EllipticEnvelope(contamination=contamination,
                             support_fraction=None, random_state=RANDOM_STATE)
        m.fit(X_scaled)
        return m.predict(X_scaled)
    raise ValueError(model_name)

tuning_results = []
for name in predictions.keys():
    for c in CONTAMINATION_GRID:
        pred = fit_and_predict(name, c)
        tn, fp, fn, tp, cost = evaluate_model(y_true, pred)
        tuning_results.append({
            "Modèle": name,
            "contamination": c,
            "TP": tp, "FP": fp, "FN": fn,
            "Recall": tp / (tp + fn) if (tp + fn) > 0 else 0,
            "Coût total (€)": cost,
        })

tuning_df = pd.DataFrame(tuning_results)
print("Tuning terminé.")


In [ ]:
# Courbes de coût en fonction de contamination, par modèle
fig, ax = plt.subplots(figsize=(11, 6))
colors = {"IsolationForest": "#4C72B0", "OneClassSVM": "#DD8452",
          "LOF": "#55A467", "EllipticEnvelope": "#C44E52"}

for name in predictions.keys():
    sub = tuning_df[tuning_df["Modèle"] == name]
    ax.plot(sub["contamination"], sub["Coût total (€)"],
            marker="o", label=name, color=colors[name], linewidth=2)
    # Marquer le minimum
    best = sub.loc[sub["Coût total (€)"].idxmin()]
    ax.scatter(best["contamination"], best["Coût total (€)"],
               s=200, color=colors[name], edgecolor="black", linewidth=2, zorder=10)

ax.axvline(0.034, color="gray", linestyle="--", alpha=0.5,
           label="Taux de panne réel (0.034)")
ax.set_xlabel("contamination / nu")
ax.set_ylabel("Coût total (€)")
ax.set_title("Coût business en fonction du seuil de contamination — par modèle")
ax.legend(loc="best")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Tableau du meilleur seuil par modèle
best_per_model = (tuning_df
                  .loc[tuning_df.groupby("Modèle")["Coût total (€)"].idxmin()]
                  .sort_values("Coût total (€)")
                  .reset_index(drop=True))

display(best_per_model[["Modèle", "contamination", "TP", "FP", "FN",
                        "Recall", "Coût total (€)"]])


### 4.4 — Décision finale & recommandation business

In [ ]:
# Modèle gagnant
winner = best_per_model.iloc[0]
print("=" * 60)
print(f"🏆 MODÈLE GAGNANT : {winner['Modèle']}")
print("=" * 60)
print(f"  Seuil optimal (contamination) : {winner['contamination']}")
print(f"  Pannes correctement détectées (TP) : {winner['TP']}")
print(f"  Pannes ratées (FN)                 : {winner['FN']}")
print(f"  Fausses alertes (FP)               : {winner['FP']}")
print(f"  Recall (taux de capture des pannes) : {winner['Recall']:.1%}")
print(f"  Coût total                         : {winner['Coût total (€)']:,.0f} €")

# Comparaison vs un baseline « do nothing » (toutes les pannes ratées)
baseline_cost = y_true.sum() * COST_FN
print(f"\nBaseline 'monitoring désactivé' : {baseline_cost:,.0f} €")
print(f"Économies réalisées par le modèle : {baseline_cost - winner['Coût total (€)']:,.0f} €")


### 4.5 — Conclusion managériale

**Synthèse pour le Comité de Direction :**

1. **Modèle recommandé** : celui qui minimise le coût total dans la grille testée (voir cellule ci-dessus).

2. **Seuil opérationnel** : `contamination` (ou `nu` pour OC-SVM) doit être réglé **au-dessus du taux de panne réel** (~3.39 %). Pourquoi ? Le ratio FN/FP de 30:1 rend l'asymétrie tellement forte qu'**il vaut mieux envoyer un technicien pour rien (500 €) que rater une panne (15 000 €)**. Le modèle doit donc être réglé en mode « parano contrôlée ».

3. **ROI estimé** : par rapport à un scénario sans monitoring (toutes les pannes coûtent 15 000 € chacune), le modèle gagnant économise plusieurs centaines de milliers d'euros par cycle de production.

4. **Prochaines étapes opérationnelles** :
   - Mettre le modèle gagnant en **production en mode shadow** pendant 1 mois (ne déclenche pas d'alerte, on observe).
   - Implémenter une **règle d'ensemble** : alerte rouge uniquement si ≥ 3/4 modèles d'accord (réduit drastiquement les FP).
   - Ré-évaluer trimestriellement, le modèle peut dériver si les machines vieillissent ou si la mix de production change (concept drift).

5. **Limites et précautions** :
   - L'évaluation a été faite sur le même dataset utilisé pour fitter → optimiste. Dans une vraie mise en production, il faut évaluer sur des données futures (out-of-time test).
   - Les 4 modèles sont entraînés sans labels ; ils ne distinguent pas entre **types** de pannes (TWF, HDF, PWF…). Pour cibler une cause racine, un modèle supervisé serait plus pertinent une fois qu'on dispose d'assez d'exemples.
